# Banded ridge for feature-specific regularization

This example combines an impulse-like feature with a smooth continuous feature. The smooth feature creates a poorly conditioned lagged design matrix, while the impulse feature remains relatively well identified. A single global ridge penalty shrinks both TRFs; `feature_alphas` lets us regularize only the smooth feature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from pyeeg.models import TRFEstimator
from pyeeg.utils import lag_matrix


In [ ]:
rng = np.random.default_rng(18)
n_samples = 2500
n_lags = 41
t = np.arange(n_samples)
impulses = (rng.random(n_samples) < 0.035).astype(float)
smooth = gaussian_filter1d(rng.standard_normal(n_samples), sigma=35)
smooth = (smooth - smooth.mean()) / smooth.std()
X = np.column_stack([impulses, smooth])
lags = np.arange(n_lags)
true_coef = np.column_stack([
    np.exp(-((lags - 8) / 3) ** 2),
    0.8 * np.exp(-((lags - 24) / 10) ** 2),
])
design = lag_matrix(X, lags=lags, mode='full', fill_value=0., block_order='features')
y = design @ true_coef + 0.35 * rng.standard_normal((n_samples, 1))
print('condition number:', np.linalg.cond(design))


In [ ]:
fig, ax = plt.subplots(2, 1, sharex=True, figsize=(10, 5))
ax[0].plot(X[:500, 0], label='impulse-like feature')
ax[1].plot(X[:500, 1], label='smooth feature', color='tab:orange')
for a in ax: a.legend(); a.grid(alpha=.2)


In [ ]:
global_ridge = TRFEstimator(times=lags[::-1], alpha=100., fit_intercept=False,
                            block_order='features', verbose=False)
banded_ridge = TRFEstimator(times=lags[::-1], feature_alphas=[0., 100.],
                            fit_intercept=False, block_order='features',
                            verbose=False)
global_ridge.fit(X, y, drop=False)
banded_ridge.fit(X, y, drop=False)


In [ ]:
fig, ax = plt.subplots(1, 2, sharey=True, figsize=(11, 4))
for feature, title, axis in zip(range(2), ['impulse-like', 'smooth'], ax):
    axis.plot(lags, true_coef[:, feature], 'k--', label='true')
    axis.plot(lags, global_ridge.coef_[:, feature, 0], label='global ridge')
    axis.plot(lags, banded_ridge.coef_[:, feature, 0], label='feature-specific ridge')
    axis.set_title(title); axis.set_xlabel('lag'); axis.grid(alpha=.2)
ax[0].set_ylabel('TRF coefficient'); ax[0].legend()


The first feature receives zero ridge while the smooth feature receives a strong penalty. In practice, choose the values in `feature_alphas` by validation or prior knowledge; this notebook deliberately fixes them to make the effect visible.